In [2]:
# ============================================================
# NOTEBOOK 01 — DATA PREPARATION
# RBI NPA Early Warning System
# ============================================================
# FILE MAP (put all files in NPA/datasets/)
#   F4_GNPA  → Other_STRBI_Table_No_20_...xlsx
#   F6_T10   → 10_Bank_Group-wise_Select_Ratios...xlsx
#   F2_ECON  → RBIB_Table_No__01___Select_Economic_Indicators.xlsx
#   F3_CRAR  → _3_Bank-wise_Capital_Adequacy_Ratios...xlsx
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = "../datasets/"   # adjust if your folder name differs

In [3]:
# ============================================================
# BLOCK 1 — GNPA DATA (F4_GNPA)
# Extract group-level rows: PUBLIC SECTOR, PRIVATE SECTOR,
# FOREIGN BANKS, SMALL FINANCE BANKS, ALL SCHEDULED
# Source: F4_GNPA → Sheet 'Report 1'
# Structure: col1=Year(filled on group total row), col2=Bank, col3=GrossNPA, col4=GrossAdvances
# ============================================================

print("=" * 55)
print("BLOCK 1: Loading F4_GNPA")
print("=" * 55)

import openpyxl

GROUP_NAMES = {
    'PUBLIC SECTOR BANKS':          'PSB',
    'PRIVATE SECTOR BANKS':         'Private',
    'FOREIGN BANKS':                'Foreign',
    'SMALL FINANCE BANKS':          'SFB',
    'ALL SCHEDULED COMMERCIAL BANKS': 'All_SCB',
}

wb = openpyxl.load_workbook(
    DATA + "F4_GNPA.xlsx", read_only=True, data_only=True
)
ws = wb['Report 1']
raw_rows = list(ws.iter_rows(values_only=True))
wb.close()

gnpa_records = []
current_year = None

for row in raw_rows:
    # col index: 0=empty, 1=Year, 2=Bank, 3=GrossNPA, 4=GrossAdvances
    if row[1] and str(row[1]).strip().isdigit():
        current_year = int(str(row[1]).strip())
    bank = str(row[2]).strip() if row[2] else ''
    if bank in GROUP_NAMES and current_year:
        try:
            gross_npa      = float(row[3]) if row[3] and str(row[3]) != '-' else np.nan
            gross_advances = float(row[4]) if row[4] and str(row[4]) != '-' else np.nan
            gnpa_records.append({
                'year':           current_year,
                'bank_group':     GROUP_NAMES[bank],
                'gross_npa_cr':   gross_npa,
                'gross_adv_cr':   gross_advances,
            })
        except:
            pass

gnpa_df = pd.DataFrame(gnpa_records)
gnpa_df = gnpa_df[gnpa_df['bank_group'] != 'All_SCB']   # keep only 4 groups
gnpa_df = gnpa_df.sort_values(['bank_group','year']).reset_index(drop=True)

# Derived columns
gnpa_df['gnpa_ratio'] = (gnpa_df['gross_npa_cr'] / gnpa_df['gross_adv_cr']) * 100

# Credit growth YoY % — computed within each bank group
gnpa_df['credit_growth'] = gnpa_df.groupby('bank_group')['gross_adv_cr'].pct_change() * 100

print(gnpa_df.pivot_table(index='year', columns='bank_group', values='gnpa_ratio').round(2))
print(f"\nGNPA rows: {len(gnpa_df)} | Years: {sorted(gnpa_df['year'].unique())}")


BLOCK 1: Loading F4_GNPA
bank_group  Foreign    PSB  Private   SFB
year                                     
2004           4.85    NaN     5.84   NaN
2005           3.05    NaN     3.83   NaN
2006           2.12    NaN     2.41   NaN
2007           1.92    NaN     2.19   NaN
2008           1.92    NaN     2.47   NaN
2009           4.37    NaN     2.92   NaN
2010           4.36    NaN     2.99   NaN
2011           2.61    NaN     2.48   NaN
2012           2.76    NaN     2.09   NaN
2013           3.04    NaN     1.77   NaN
2014           3.86    NaN     1.78   NaN
2015           3.20    NaN     2.10   NaN
2016           4.20    NaN     2.83   NaN
2017           3.96    NaN     4.05   NaN
2018           3.81  14.58     4.62  2.53
2019           2.99  11.59     5.25  1.83
2020           2.34  10.25     5.45  1.87
2021           2.42   9.11     4.94  5.35
2022           2.90   7.28     3.84  4.94
2023           1.89   4.97     2.29  4.81
2024           1.19   3.47     1.85  2.43
2025     

In [4]:
# ============================================================
# BLOCK 2 — ROA FROM F6_T10
# Structure: every 29 rows = one year
# Row 28 (0-indexed 27) in each block = ROA row
# Columns: 4=PSB(SBI+Assoc), 5=Nationalised, 6=PSB_total,
#          7=Private, 8=Foreign, 9=SFB, 10=Payments, 11=All_SCB
# We use: col6=PSB, col7=Private, col8=Foreign, col9=SFB
# ============================================================

print("\n" + "=" * 55)
print("BLOCK 2: Loading F6_T10 → ROA by bank group")
print("=" * 55)

wb2 = openpyxl.load_workbook(
    DATA + "F6_T10.xlsx", read_only=True, data_only=True
)
ws2 = wb2['Report 1']
t10_rows = list(ws2.iter_rows(values_only=True))
wb2.close()

# Year appears in col index 2, every 29 rows starting at row 8 (0-idx 7)
# ROA is ratio #21 → offset 20 from the year row (0-indexed within block)
roa_records = []
current_yr = None

for i, row in enumerate(t10_rows):
    # Capture year
    yr_raw = str(row[2]).strip() if row[2] else ''
    if '-' in yr_raw and len(yr_raw) == 7:   # e.g. '2024-25'
        current_yr = int(yr_raw[:4]) + 1      # 2024-25 → fiscal year end = 2025

    # ROA row identified by ratio label
    if row[3] and 'Return on assets' in str(row[3]) and current_yr:
        try:
            roa_records.append({
                'year':    current_yr,
                'PSB':     float(row[6])  if row[6]  and str(row[6]).strip()  not in ['-',' ',''] else np.nan,
                'Private': float(row[7])  if row[7]  and str(row[7]).strip()  not in ['-',' ',''] else np.nan,
                'Foreign': float(row[8])  if row[8]  and str(row[8]).strip()  not in ['-',' ',''] else np.nan,
                'SFB':     float(row[9])  if row[9]  and str(row[9]).strip()  not in ['-',' ',''] else np.nan,
            })
        except:
            pass

roa_wide = pd.DataFrame(roa_records).drop_duplicates(subset='year')
# Melt to long format matching gnpa_df structure
roa_long = roa_wide.melt(id_vars='year', var_name='bank_group', value_name='roa')
print(roa_wide.set_index('year').round(3))
print(f"\nROA rows: {len(roa_wide)} years")



BLOCK 2: Loading F6_T10 → ROA by bank group
        PSB  Private  Foreign    SFB
year                                
2025  1.096    1.778    1.696  0.968
2024  0.957    1.868    1.654  2.117
2023  0.786    1.633    1.473  1.802
2022  0.547    1.451    1.414  0.534
2021  0.281    1.168    1.559  1.362
2020 -0.232    0.513    1.546  1.706
2019 -0.651    0.630    1.565  1.587
2018 -0.843    1.140    1.343  0.732
2017 -0.101    1.299    1.612  1.130
2016 -0.068    1.496    1.448    NaN
2015  0.460    1.676    1.840    NaN
2014  0.504    1.653    1.540    NaN
2013  0.796    1.634    1.917    NaN
2012  0.884    1.531    1.762    NaN
2011  0.959    1.426    1.745    NaN
2010  0.970    1.277    1.263    NaN
2009  1.025    1.128    1.989    NaN
2008  0.998    1.131    2.089    NaN
2007  0.918    1.025    2.285    NaN
2006  0.882    1.070    2.081    NaN
2005  0.951    1.055    1.605    NaN

ROA rows: 21 years


In [5]:
# ============================================================
# BLOCK 3 — MACRO VARIABLES FROM F2_ECON
# Monthly sheet:
#   col index 1  = Date
#   col index 2  = IIP growth %
#   col index 4  = SCB Credit growth %
#   col index 16 = Policy Repo Rate %
#   col index 34 = CPI % change
# Quarterly sheet:
#   col index 1  = Year
#   col index 2  = Quarter
#   col index 3  = GVA/GDP growth %
# ============================================================

print("\n" + "=" * 55)
print("BLOCK 3: Loading F2_ECON → Macro variables")
print("=" * 55)

wb3 = openpyxl.load_workbook(
    DATA + "F2_ECON.xlsx", read_only=True, data_only=True
)

# --- Monthly sheet → repo rate, CPI, IIP, credit growth ---
ws_m = wb3['Monthly']
monthly_rows = list(ws_m.iter_rows(values_only=True))

monthly_records = []
for row in monthly_rows[7:]:    # data starts row 8 (0-idx 7)
    try:
        date_val = row[1]
        if date_val is None:
            continue
        # date_val is a datetime object from openpyxl
        if hasattr(date_val, 'year'):
            dt = date_val
        else:
            dt = pd.to_datetime(str(date_val), errors='coerce')
            if pd.isnull(dt):
                continue

        def safe_float(v):
            try:
                f = float(v)
                return f if abs(f) < 1e6 else np.nan
            except:
                return np.nan

        monthly_records.append({
            'date':          pd.Timestamp(dt.year, dt.month, 1),
            'iip_growth':    safe_float(row[2]),
            'credit_growth_scb': safe_float(row[4]),   # SCB credit YoY%
            'repo_rate':     safe_float(row[16]),
            'cpi_yoy':       safe_float(row[34]),
        })
    except:
        pass

monthly_df = pd.DataFrame(monthly_records).dropna(subset=['date'])
monthly_df = monthly_df.sort_values('date').drop_duplicates('date')

# Convert to fiscal year (April–March): fiscal year = year of March end
monthly_df['fiscal_year'] = monthly_df['date'].apply(
    lambda d: d.year if d.month <= 3 else d.year + 1
)

macro_annual = monthly_df.groupby('fiscal_year').agg(
    repo_rate      = ('repo_rate',        'mean'),
    cpi_avg        = ('cpi_yoy',          'mean'),
    iip_avg        = ('iip_growth',       'mean'),
    credit_growth_scb = ('credit_growth_scb','mean'),
).round(3).reset_index()
macro_annual = macro_annual.rename(columns={'fiscal_year': 'year'})

# --- Quarterly sheet → annual GDP growth ---
ws_q = wb3['Quarterly']
quarterly_rows = list(ws_q.iter_rows(values_only=True))

gdp_records = []
current_fy = None
for row in quarterly_rows[5:]:
    yr_raw = str(row[1]).strip() if row[1] else ''
    if '-' in yr_raw and len(yr_raw) == 7:
        current_fy = int(yr_raw[:4]) + 1    # 2025-26 → 2026
    try:
        gdp_val = float(row[3])
        if current_fy and not np.isnan(gdp_val):
            gdp_records.append({'year': current_fy, 'gdp_growth': gdp_val})
    except:
        pass

gdp_df = pd.DataFrame(gdp_records).groupby('year')['gdp_growth'].mean().round(3).reset_index()

# Merge GDP into macro_annual
macro_annual = pd.merge(macro_annual, gdp_df, on='year', how='left')
wb3.close()

print(macro_annual.tail(10).to_string(index=False))
print(f"\nMacro annual rows: {len(macro_annual)}")


BLOCK 3: Loading F2_ECON → Macro variables
 year  repo_rate  cpi_avg  iip_avg  credit_growth_scb  gdp_growth
 2018      6.083    4.491    4.418              7.245       6.218
 2019      6.333    4.197    3.914             13.191       5.833
 2020      5.371    4.249   -0.639              9.333       3.951
 2021      4.033    4.287   -8.202              5.830      -4.268
 2022      4.000    4.462    7.718              7.308      10.252
 2023      5.567    2.810    5.513             15.250       6.962
 2024      6.500    1.530    5.901             18.865       8.189
 2025      6.458    1.895    4.685             13.631       7.005
 2026      5.500    2.308    4.116             11.739       8.034
 2027      5.250    3.282    4.900             16.001         NaN

Macro annual rows: 18


In [6]:
# ============================================================
# BLOCK 4 — MERGE INTO PANEL DATASET
# ============================================================

print("\n" + "=" * 55)
print("BLOCK 4: Building panel dataset")
print("=" * 55)

# Merge GNPA + ROA
panel = pd.merge(gnpa_df, roa_long, on=['year','bank_group'], how='left')

# Merge macro (same macro for all bank groups in same year)
panel = pd.merge(panel, macro_annual, on='year', how='left')

# Remove credit_growth_scb from macro (we already have group-wise credit_growth from GNPA)
panel = panel.drop(columns=['credit_growth_scb'], errors='ignore')

# Sort properly
panel = panel.sort_values(['bank_group','year']).reset_index(drop=True)


BLOCK 4: Building panel dataset


In [7]:
# ============================================================
# BLOCK 5 — DERIVED COLUMNS
# PCR = 1 − (Net NPA / Gross NPA)
# We don't have Net NPA in this file — so we use the
# RBI standard approximation: PCR proxy from provisioning
# PCR will be added manually from RTP PDF data below.
# For now, flag as NaN — add after manual copy from PDF.
# ============================================================

panel['pcr'] = np.nan    # PLACEHOLDER — fill from RTP PDF manually

# Lag features — credit growth 1 and 2 years prior
panel['credit_growth_lag1'] = panel.groupby('bank_group')['credit_growth'].shift(1)
panel['credit_growth_lag2'] = panel.groupby('bank_group')['credit_growth'].shift(2)

# 3-year rolling avg credit growth
panel['credit_growth_3y_avg'] = panel.groupby('bank_group')['credit_growth'].transform(
    lambda x: x.rolling(3, min_periods=2).mean()
)

# Target variables — GNPA ratio 1 and 2 years ahead
panel['gnpa_next1yr'] = panel.groupby('bank_group')['gnpa_ratio'].shift(-1)
panel['gnpa_next2yr'] = panel.groupby('bank_group')['gnpa_ratio'].shift(-2)

# Binary stress flags
panel['stress_next1yr'] = (panel['gnpa_next1yr'] > 7.0).astype(float)
panel['stress_next2yr'] = (panel['gnpa_next2yr'] > 7.0).astype(float)
# NaN rows get NaN not 0
panel.loc[panel['gnpa_next1yr'].isna(), 'stress_next1yr'] = np.nan
panel.loc[panel['gnpa_next2yr'].isna(), 'stress_next2yr'] = np.nan


In [8]:
# ============================================================
# BLOCK 6 — QUALITY CHECKS
# ============================================================

print("\n" + "=" * 55)
print("BLOCK 6: Quality checks")
print("=" * 55)

print(f"\nPanel shape: {panel.shape}")
print(f"Bank groups: {sorted(panel['bank_group'].unique())}")
print(f"Year range:  {panel['year'].min()} – {panel['year'].max()}")

print("\nNull counts per column:")
print(panel.isnull().sum())

print("\nSample — PSB rows:")
print(panel[panel['bank_group']=='PSB'][[
    'year','bank_group','gnpa_ratio','credit_growth',
    'credit_growth_lag2','roa','repo_rate','gdp_growth',
    'stress_next1yr'
]].to_string(index=False))


BLOCK 6: Quality checks

Panel shape: (60, 19)
Bank groups: ['Foreign', 'PSB', 'Private', 'SFB']
Year range:  2004 – 2025

Null counts per column:
year                     0
bank_group               0
gross_npa_cr             0
gross_adv_cr             0
gnpa_ratio               0
credit_growth            4
roa                      2
repo_rate               14
cpi_avg                 14
iip_avg                 14
gdp_growth              18
pcr                     60
credit_growth_lag1       8
credit_growth_lag2      12
credit_growth_3y_avg     8
gnpa_next1yr             4
gnpa_next2yr             8
stress_next1yr           4
stress_next2yr           8
dtype: int64

Sample — PSB rows:
 year bank_group  gnpa_ratio  credit_growth  credit_growth_lag2       roa  repo_rate  gdp_growth  stress_next1yr
 2018        PSB   14.582307            NaN                 NaN -0.843000      6.083       6.218             1.0
 2019        PSB   11.587082       3.920132                 NaN -0.650992      6

In [10]:
# ============================================================
# BLOCK 7 — SAVE
# ============================================================

panel.to_csv(DATA + "banking_panel.csv", index=False)
print("\n✅ Saved → datasets/banking_panel.csv")

macro_annual.to_csv(DATA + "macro_annual.csv", index=False)
print("✅ Saved → datasets/macro_annual.csv")

gnpa_df.to_csv(DATA + "gnpa_clean.csv", index=False)
print("✅ Saved → datasets/gnpa_clean.csv")

print("\n" + "=" * 55)
print("NOTEBOOK 01 COMPLETE")
print("=" * 55)
print("""
WHAT YOU HAVE NOW:
  banking_panel.csv  — main panel (year × bank_group)
  macro_annual.csv   — GDP, repo rate, CPI, IIP annual
  gnpa_clean.csv     — GNPA + Gross Advances by group

WHAT STILL NEEDS MANUAL INPUT:
  panel['pcr'] — open your RTP 2024-25 PDF
                 Ctrl+F: 'Provision Coverage Ratio'
                 Copy the bank-group table into:
                 datasets/pcr_manual.csv
                 columns: year, PSB, Private, Foreign, SFB
                 Then re-run from BLOCK 4 with:
                 pcr_df = pd.read_csv('datasets/pcr_manual.csv')
                 pcr_long = pcr_df.melt(id_vars='year',
                             var_name='bank_group', value_name='pcr')
                 panel = pd.merge(panel.drop('pcr',axis=1),
                                  pcr_long, on=['year','bank_group'], how='left')
""")


✅ Saved → datasets/banking_panel.csv
✅ Saved → datasets/macro_annual.csv
✅ Saved → datasets/gnpa_clean.csv

NOTEBOOK 01 COMPLETE

WHAT YOU HAVE NOW:
  banking_panel.csv  — main panel (year × bank_group)
  macro_annual.csv   — GDP, repo rate, CPI, IIP annual
  gnpa_clean.csv     — GNPA + Gross Advances by group

WHAT STILL NEEDS MANUAL INPUT:
  panel['pcr'] — open your RTP 2024-25 PDF
                 Ctrl+F: 'Provision Coverage Ratio'
                 Copy the bank-group table into:
                 datasets/pcr_manual.csv
                 columns: year, PSB, Private, Foreign, SFB
                 Then re-run from BLOCK 4 with:
                 pcr_df = pd.read_csv('datasets/pcr_manual.csv')
                 pcr_long = pcr_df.melt(id_vars='year',
                             var_name='bank_group', value_name='pcr')
                 panel = pd.merge(panel.drop('pcr',axis=1),
                                  pcr_long, on=['year','bank_group'], how='left')



In [11]:
# ============================================================
# BLOCK 8 — PCR FROM F7_NPA_MOVEMENT
# Add this to the end of notebook 01 (after BLOCK 7)
# or run standalone — it reads banking_panel.csv and updates it
#
# F7 column layout (0-indexed):
#   col 1 = Year (filled only on ALL SCHEDULED row)
#   col 2 = Bank name
#   col 3 = Gross NPA Opening Balance
#   col 7 = Gross NPA Closing Balance   ← not needed (have from F4)
#   col 8 = Net NPA Opening Balance
#   col 9 = Net NPA Closing Balance     ← THIS is what we need
#
# PCR = (Gross NPA - Net NPA) / Gross NPA * 100
# Using CLOSING balances for both (end of March each year)
# ============================================================

import pandas as pd
import numpy as np
import openpyxl

DATA = "../datasets/"

GROUP_MAP = {
    'PUBLIC SECTOR BANKS':            'PSB',
    'PRIVATE SECTOR BANKS':           'Private',
    'FOREIGN BANKS':                  'Foreign',
    'SMALL FINANCE BANKS':            'SFB',
    'ALL SCHEDULED COMMERCIAL BANKS': 'All_SCB',
}

# ── Load F7 ──────────────────────────────────────────────────
wb = openpyxl.load_workbook(
    DATA + "F7_NPA_MOVEMENT.xlsx", read_only=True, data_only=True
)
ws = wb['Report 1']
rows = list(ws.iter_rows(values_only=True))
wb.close()

# ── Extract Net NPA closing balance by group & year ──────────
net_npa_records = []
current_year = None

for row in rows:
    # Year is filled only on the ALL SCHEDULED row
    if row[1] and str(row[1]).strip().isdigit():
        current_year = int(str(row[1]).strip())

    bank = str(row[2]).strip() if row[2] else ''

    if bank in GROUP_MAP and current_year and bank != 'ALL SCHEDULED COMMERCIAL BANKS':
        def safe(v):
            try:
                f = float(v)
                return f if f > 0 else np.nan
            except:
                return np.nan

        gross_npa_closing = safe(row[7])   # col index 7
        net_npa_closing   = safe(row[9])   # col index 9

        net_npa_records.append({
            'year':             current_year,
            'bank_group':       GROUP_MAP[bank],
            'gross_npa_f7_cr':  gross_npa_closing,
            'net_npa_cr':       net_npa_closing,
        })

net_npa_df = pd.DataFrame(net_npa_records)
net_npa_df = net_npa_df.sort_values(['bank_group','year']).reset_index(drop=True)

print("Net NPA closing balance extracted:")
print(net_npa_df.pivot_table(
    index='year', columns='bank_group', values='net_npa_cr'
).round(0).to_string())

# ── Load existing panel and merge ────────────────────────────
panel = pd.read_csv(DATA + "banking_panel.csv")

# Drop placeholder pcr column
panel = panel.drop(columns=['pcr'], errors='ignore')

# Merge net NPA
panel = pd.merge(
    panel,
    net_npa_df[['year','bank_group','net_npa_cr']],
    on=['year','bank_group'],
    how='left'
)

# ── Compute PCR ──────────────────────────────────────────────
# Use gross_npa_cr from panel (sourced from F4_GNPA — more complete)
panel['pcr'] = (
    (panel['gross_npa_cr'] - panel['net_npa_cr'])
    / panel['gross_npa_cr']
) * 100

# Sanity check — PCR must be between 0 and 100
panel.loc[panel['pcr'] < 0,   'pcr'] = np.nan
panel.loc[panel['pcr'] > 100, 'pcr'] = np.nan

# ── Quality check ────────────────────────────────────────────
print("\nPCR by bank group and year:")
print(panel.pivot_table(
    index='year', columns='bank_group', values='pcr'
).round(1).to_string())

print("\nNull count in pcr:", panel['pcr'].isna().sum())
print("PCR range:", round(panel['pcr'].min(),1), "–", round(panel['pcr'].max(),1))

# ── Save updated panel ───────────────────────────────────────
panel.to_csv(DATA + "banking_panel.csv", index=False)
print("\n✅ banking_panel.csv updated with PCR column")
print(f"   Panel shape: {panel.shape}")
print(f"   Columns: {list(panel.columns)}")

Net NPA closing balance extracted:
bank_group  Foreign       PSB  Private     SFB
year                                          
2005          639.0       NaN   4212.0     NaN
2006          808.0       NaN   3170.0     NaN
2007          927.0       NaN   4028.0     NaN
2008         1247.0       NaN   5647.0     NaN
2009         2997.0       NaN   7412.0     NaN
2010         2977.0       NaN   6506.0     NaN
2011         1312.0       NaN   4432.0     NaN
2012         1412.0       NaN   4401.0     NaN
2013         2663.0       NaN   5994.0     NaN
2014         3160.0       NaN   8862.0     NaN
2015         1762.0       NaN  14128.0     NaN
2016         2762.0       NaN  26677.0     NaN
2017         2137.0       NaN  47780.0   115.0
2018         1548.0  454473.0  64380.0   436.0
2019         2051.0  285122.0  67309.0   586.0
2020         2005.0  230918.0  55683.0   765.0
2021         3241.0  196451.0  55377.0  2981.0
2022         3023.0  154745.0  43738.0  2725.0
2023         1656.0  1025